# Session 6 — Demo 2 — Why models lie: leakage and the lucky split

**What this demo covers.** Two ways a beautiful test number turns out to be fiction:

1. **Leakage:** a column, or a step of your *procedure*, that secretly saw the answer;
2. **The lucky split:** one test number is itself a random variable.

Both are shown on data where we know the truth, and both have procedural cures. The grade throughout is AUC from Demo 1: 0.5 is a coin flip, 1.0 is a perfect order.

**Data.** `data/bankruptcy/taiwan_bankruptcy.csv`.

**How to run.** Locally: top to bottom. Colab: uncomment the two `git clone` lines **and** the `%pip install catboost` line in the setup cell: `catboost` is not preinstalled in Colab.

## 0. Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# the Taiwan bankruptcy dataset (sessions 5 to 7)
BANKRUPTCY = Path("../../data/bankruptcy")

# --- Google Colab? Uncomment these lines to fetch the course repo: ---
# !git clone https://github.com/seemsGoodNow/hse-data-science-course.git course
# BANKRUPTCY = Path("course/data/bankruptcy")
# %pip install catboost

In [2]:
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

firms = pd.read_csv(BANKRUPTCY / "taiwan_bankruptcy.csv")
firms.columns = firms.columns.str.strip()
firms = firms.drop(columns=["Net Income Flag"])

y = firms["Bankrupt?"]
X = firms.drop(columns=["Bankrupt?"])

## 1. The crude leak, staged

We add one column that was *built from the answer*. Here we do it openly: `audit_flag` is about 0.9 for every bankrupt firm and about 0 for every healthy one, with a little random noise so that it is not an exact copy. In real data such a column walks in disguised: "date the debt was written off" among default features, "payout amount" among churn features, an "audit result" logged *after* the bankruptcy.

`X.copy()` makes a separate table, so the new column doesn't land in `X`.

In [3]:
rng = np.random.default_rng(1)

leaky_X = X.copy()
leaky_X["audit_flag"] = y * 0.9 + rng.normal(0, 0.1, len(y))

leaky_X_train, leaky_X_test, y_train, y_test = train_test_split(
    leaky_X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y,
)
leaky_model = CatBoostClassifier(
    iterations=100,
    random_seed=42,
    verbose=0,
)
leaky_model.fit(leaky_X_train, y_train)

leaky_auc = roc_auc_score(y_test, leaky_model.predict_proba(leaky_X_test)[:, 1])
print(f"test AUC with the leak: {leaky_auc:.4f}")

test AUC with the leak: 1.0000


**Test AUC 1.0000, and the split didn't save us:** the leak sits in both halves. The rule with no exceptions: *a result too good to be true IS true, so go and hunt the leak.*

**How to hunt a leak in a table you didn't build.** Two steps.

1. **Ask the documentation one question about every column:** is it known at the moment of the decision, or only after the outcome? A column filled in after the outcome must go, however good it makes the model look.
2. **If a model scores suspiciously high, open its importance list** (session 5). A leaked column sits at the top. Try it yourself 1 below does exactly that.

In HW2 this is Part 0. If you did Part 0 before tonight, reopen your list with this question.

## 2. The subtle leak: selection before the split

Now the version that catches professionals. A boutique setting: **240 firms**, the size of a due-diligence sample, and **500 candidate "indicators" that are pure noise**. We generate them ourselves, so we KNOW they say nothing about bankruptcy.

`.sample(n=240)` picks 240 random firms (session 2). `rng.normal(size=(240, 500))` draws a whole table of random numbers at once, 240 rows by 500 columns. `.add_prefix("indicator_")` names the columns `indicator_0`, `indicator_1`, and so on.

In [4]:
small_firms = firms.sample(n=240, random_state=7).reset_index(drop=True)
y_small = small_firms["Bankrupt?"]

noise = pd.DataFrame(rng.normal(size=(240, 500))).add_prefix("indicator_")
print(
    f"{len(small_firms)} firms, {y_small.sum()} bankrupt, "
    f"{noise.shape[1]} candidate indicators, all pure noise"
)

240 firms, 11 bankrupt, 500 candidate indicators, all pure noise


The tempting pipeline: *"screen the indicators first, keep the 10 most correlated with bankruptcy, then split and model properly."* Sounds diligent. Step 1 is the screening. `.corrwith(y_small)` is the `.corr()` you know, for every column against one target at once. `.abs()` drops the sign, because a strong minus is as useful as a strong plus.

In [5]:
correlation_with_bankruptcy = (
    noise
    .corrwith(y_small)
    .abs()
    .sort_values(ascending=False)
)
correlation_with_bankruptcy.head(10).round(3)

indicator_446    0.205
indicator_60     0.184
indicator_421    0.182
indicator_282    0.171
indicator_34     0.169
indicator_126    0.167
indicator_2      0.165
indicator_98     0.158
indicator_360    0.157
indicator_4      0.157
dtype: float64

Correlations up to 0.2, **from pure noise**. With 500 tries, a few columns line up with bankruptcy by chance.

Two functions run the two orders on one split and return the test AUC. Read them step by step:

- **select, then split:** the top ten indicators are chosen on all 240 firms; then the split; then a logistic regression on the training part, graded on the test part;
- **split, then select:** the split comes first, and the top ten are chosen on the training part only. The test firms never vote on which indicators to keep.

`seed` goes into `random_state`, so each seed gives a different split.

In [6]:
def score_select_then_split(seed):
    """Test AUC when the top ten indicators are chosen on all 240 firms."""
    top_ten = correlation_with_bankruptcy.head(10).index
    noise_train, noise_test, y_noise_train, y_noise_test = train_test_split(
        noise[top_ten],
        y_small,
        test_size=0.4,
        random_state=seed,
        stratify=y_small,
    )
    model = LogisticRegression()
    model.fit(noise_train, y_noise_train)
    return roc_auc_score(y_noise_test, model.predict_proba(noise_test)[:, 1])


def score_split_then_select(seed):
    """Test AUC when the top ten indicators are chosen on the training part only."""
    noise_train, noise_test, y_noise_train, y_noise_test = train_test_split(
        noise,
        y_small,
        test_size=0.4,
        random_state=seed,
        stratify=y_small,
    )
    top_ten = (
        noise_train
        .corrwith(y_noise_train)
        .abs()
        .sort_values(ascending=False)
        .head(10)
        .index
    )
    model = LogisticRegression()
    model.fit(noise_train[top_ten], y_noise_train)
    return roc_auc_score(y_noise_test, model.predict_proba(noise_test[top_ten])[:, 1])


print(
    f"one split, select THEN split: test AUC {score_select_then_split(0):.2f}\n"
    f"one split, split THEN select: test AUC {score_split_then_select(0):.2f}"
)

one split, select THEN split: test AUC 0.83
one split, split THEN select: test AUC 0.70


One split each. But the test part holds only 4 bankrupt firms, so one number means little. Let's repeat both orders on ten different splits, seeds 0 to 9. `np.round` rounds a whole list at once, and `np.mean` is its average.

In [7]:
select_first_aucs = []
split_first_aucs = []
for seed in range(10):
    select_first_aucs.append(score_select_then_split(seed))
    split_first_aucs.append(score_split_then_select(seed))

print(
    f"select THEN split: {np.round(select_first_aucs, 2)}, "
    f"mean {np.mean(select_first_aucs):.2f}\n"
    f"split THEN select: {np.round(split_first_aucs, 2)}, "
    f"mean {np.mean(split_first_aucs):.2f}"
)

select THEN split: [0.83 0.85 0.85 0.82 0.95 0.93 0.82 0.81 0.78 0.96], mean 0.86
split THEN select: [0.7  0.7  0.8  0.54 0.46 0.43 0.46 0.53 0.44 0.38], mean 0.54


**The tempting order manufactures a strong model out of nothing:** a mean AUC of 0.86, single splits up to 0.96. The honest order gives 0.54, about what noise deserves. Look at the honest row too. One split scored 0.80 from pure noise, another 0.38. With 4 bankrupt firms in the test part, one number swings wildly: hold that thought for section 3.

Nothing "leaked" in the obvious sense. **The procedure peeked**: choosing the indicators on all firms let the test firms vote, and their chance correlations came along.

The cure, in one sentence: **everything that computes something from the data (a mean and std for standardizing, as in session 5; a list of top features) is part of the model, and lives strictly inside the training part of the split.** On small financial samples this is the classic self-deception.

## 3. The lucky split → cross-validation

Even with an honest pipeline, ONE test number is a random variable, especially with 3% bankrupt firms. **Cross-validation** (CV) measures how much it moves.

`cv=5` cuts the firms into five equal parts, called **folds**, and runs five rounds:

```
round 1:  [test]  train   train   train   train
round 2:   train [test]   train   train   train
round 3:   train  train  [test]   train   train
round 4:   train  train   train  [test]   train
round 5:   train  train   train   train  [test]
```

Each fold plays the test once; the other four train the model. Every firm is tested exactly once, and never by a model that saw it. For a classifier the folds keep the bankrupt share, like `stratify`. `cross_val_score` runs the five rounds and returns five scores. `scoring=` picks the grade: `"roc_auc"`, `"recall"`, `"precision"` and others.

Two more tools. `StandardScaler` does session 5's `(X - train_mean) / train_std`: `.fit` learns the mean and std, `.transform` applies them. `make_pipeline` glues the scaler and the model into one object. Inside every CV round the scaler then learns from that round's training part only: section 2's lesson, built in.

Our five-ratio logistic regression from session 5, five folds:

In [8]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

FIVE_RATIOS = [
    "Working Capital to Total Assets",
    "Retained Earnings to Total Assets",
    "ROA(C) before interest and depreciation before interest",
    "Total Asset Turnover",
    "Debt ratio %",
]
scaled_logit = make_pipeline(StandardScaler(), LogisticRegression())
fold_aucs = cross_val_score(
    scaled_logit,
    X[FIVE_RATIOS],
    y,
    cv=5,
    scoring="roc_auc",
)
print(
    f"AUC by fold: {fold_aucs.round(3)}\n"
    f"mean {fold_aucs.mean():.3f} ± {fold_aucs.std():.3f}"
)

AUC by fold: [0.951 0.842 0.939 0.915 0.959]
mean 0.921 ± 0.043


**0.84 to 0.96, depending on which fifth of the data played the test.** One split could have handed us any of those numbers, and we would have believed it. The ± is the std of the five fold scores (session 3): a typical distance of one fold from the mean, not a guaranteed range.

The discipline:

- **cross-validation to choose** (models, features, settings): it averages the luck out;
- **the test part touched once**, at the very end, for the number you report.

Here we ran CV on all firms to show the spread. When CV chooses something, run it on the training part only, and keep the test part for the final number.

For HW2: what matters is how many "yes" rows land in your test part. The fewer there are, the more your numbers wobble, and CV is the honest way to say so. Report the spread in the score you compare models by (`scoring="recall"`, `"precision"` or `"roc_auc"`), and say which. On data with text columns, CatBoost needs them inside CV too: `cross_val_score(..., params={"cat_features": text_columns})`.

## Before you believe a model: the checklist

1. Split first; **everything that learns lives inside the training part**: selection, scaling.
2. Train-vs-test gap checked, the memorization alarm (session 5: 154 of 154 on train, 34 of 66 on test; HW2 Part 4 asks for the same).
3. The metric fits the class balance: AUC, recall and precision, not accuracy.
4. Threshold chosen by **costs**, and its price shown against 0.5.
5. Every column known before the outcome; too good to be true means a leak hunt, starting at the top of feature importance.
6. Small data: cross-validate to choose, touch the test once. (Demo 1's threshold broke this rule a little, on purpose and out loud.)

*Items 1 to 5 are the correctness row of the HW2 rubric.*

## Try it yourself

1. Find the leak like an auditor: print the leaky model's top-5 feature importances (`get_feature_importance()`, session 5). Where does `audit_flag` rank?
2. Shrink the boutique to 120 firms. Does the tempting order get better or worse? Why?
3. Stretch: run `cross_val_score` on the full boosting model (fewer iterations, and patience). How wide are ITS folds? On HW2 data with text columns, remember `params={"cat_features": text_columns}`.

In [9]:
# your turn

## Recap

- Crude leak: a column built from the answer gives test AUC 1.0, and the split can't save you. Too good to be true is true.
- Hunt leaks with one question per column (known before the outcome, or only after?), then check the top of the importance list.
- Subtle leak: **selection before the split** turns 500 noise indicators into a model with AUC 0.86 on 240 firms. The procedure peeked.
- Everything that learns from data is part of the model and lives inside the training part; pipelines make it automatic.
- One test number is a random variable (folds from 0.84 to 0.96). CV to choose, the test once.

*HW2 is due Oct 21, 18:00. Next week, the finale: clustering real bank-customer complaints, and the full loop from raw data to an AI-built HTML report.*